# Pipeline SFT — Logic_Based_Educational_Queries

Supervised fine-tune **Qwen2.5-7B-Instruct** + **LoRA r=8** + **bf16**.

- Flatten 808 câu từ 411 record (giống `eda_and_preprocessing.ipynb`)
- Chia **train / dev / test = 8:1:1** theo `record_id`
- Push lên HF: `Laplaces-Red-Devils/logic-v01-fewshot-qwen2.5-7B`

**Chỉnh prompt & hyperparams:** cell **Cấu hình** bên dưới.

## 0. Cài dependency (chạy 1 lần)

In [22]:
%pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 3.0 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.1 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [24]:
%pip uninstall -y torchao
%pip install "torchao>=0.16.0"

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 8.2 MB/s eta 0:00:0000:0100:01


In [ ]:
# %pip install -q transformers datasets peft "trl>=0.16.0" accelerate bitsandbytes scikit-learn huggingface_hub gdown

## 1. Cấu hình — chỉnh tại đây

In [12]:
from __future__ import annotations

import json
import os
from pathlib import Path

# --- Model ---
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
MAX_SEQ_LENGTH = 2048
TRUST_REMOTE_CODE = True

# --- LoRA ---
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# --- Data ---
# "local" = file có sẵn trong repo | "drive" = tải zip Google Drive rồi giải nén
DATA_SOURCE = "drive"  # Colab: "drive" | máy có repo: "local"
DATA_FILENAME = "Logic_Based_Educational_Queries.json"
GDRIVE_FILE_ID = "1-8Wu-2FikPQIX7_rRiY3wfe7c229To2Z"
GDRIVE_ZIP_NAME = "EXACT2026_dataset_2026-05-15.zip"
GDRIVE_URL = "https://drive.google.com/file/d/1-8Wu-2FikPQIX7_rRiY3wfe7c229To2Z/view?usp=drive_link"
DATA_ROOT = Path.cwd() / "exact_data"  # Colab: /content/exact_data

SPLIT_RATIOS = (0.8, 0.1, 0.1)  # train, dev, test
SPLIT_SEED = 42
INCLUDE_FOL_IN_USER = True
# Thư mục gốc chứa train/ dev/ test/ (mỗi split một folder)
SPLITS_EXPORT_DIR = None  # None → tự đặt sau khi resolve DATA_PATH (xem cell export)
EXPORT_SPLITS = True  # False nếu đã export rồi, không ghi lại

# --- Prompt (chỉnh tự do) ---
SYSTEM_PROMPT = """\
You are an expert in formal logic and educational reasoning. Given premises in natural language (and optionally first-order logic), answer the question with the correct label and a clear explanation that cites relevant premises when appropriate.
"""

USER_TEMPLATE = """\
{premises_block}

Question:
{question}
"""

ASSISTANT_TEMPLATE = """\
Answer: {answer}

Explanation: {explanation}
"""

PREMISES_FOL_HEADER = "Premises (FOL):"
PREMISES_NL_HEADER = "Premises (NL):"

# --- GPU / train mode ---
# "auto" | "kaggle_p100" (QLoRA 4-bit, fp16) | "default" (bf16 LoRA, T4/A100)
GPU_PROFILE = "auto"

# --- Training ---
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
LOGGING_STEPS = 10
BF16 = True
GRADIENT_CHECKPOINTING = True  # False nếu vẫn CheckpointError / thiếu VRAM thì giảm batch
TRAIN_SEED = 3407
RUN_TRAIN = True  # False nếu chỉ chuẩn bị data / xem mẫu

# --- Hugging Face Hub ---
HF_REPO_ID = "Laplaces-Red-Devils/logic-v01-fewshot-qwen2.5-7B"
HF_PRIVATE = False
HF_COMMIT_MESSAGE = "logic-v01 few-shot SFT on Logic_Based_Educational_Queries"
PUSH_TO_HUB = False  # True sau khi train xong + đã set HF_TOKEN

print("Config OK")

Config OK


## 2. Đường dẫn & flatten + split 8:1:1

Đặt `DATA_SOURCE` ở cell cấu hình:
- **`local`**: dùng `app/data/raw/...` trong repo (clone / máy local)
- **`drive`**: tải [EXACT2026_dataset zip](https://drive.google.com/file/d/1-8Wu-2FikPQIX7_rRiY3wfe7c229To2Z/view?usp=drive_link) → giải nén → copy JSON vào `DATA_ROOT/app/data/raw/`

In [13]:
import shutil
import subprocess
import sys
import zipfile


def _ensure_gdown():
    try:
        import gdown  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])


def download_and_extract_from_drive() -> Path:
    """Tải zip Drive, giải nén, tìm Logic JSON, copy về DATA_ROOT/app/data/raw/."""
    target = (DATA_ROOT / "app" / "data" / "raw" / DATA_FILENAME).resolve()
    if target.is_file():
        print(f"Đã có sẵn (bỏ qua tải): {target}")
        return target

    cache = (DATA_ROOT / "_cache").resolve()
    cache.mkdir(parents=True, exist_ok=True)
    zip_path = cache / GDRIVE_ZIP_NAME

    _ensure_gdown()
    import gdown

    if not zip_path.is_file():
        print(f"Tải Google Drive ({GDRIVE_ZIP_NAME})...")
        print(f"  Link: {GDRIVE_URL}")
        gdown.download(
            f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}",
            str(zip_path),
            quiet=False,
        )
    else:
        print(f"Zip đã có: {zip_path}")

    extract_dir = cache / "extracted"
    marker = extract_dir / ".extracted"
    if not marker.exists():
        extract_dir.mkdir(parents=True, exist_ok=True)
        print(f"Giải nén → {extract_dir}")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_dir)
        marker.write_text("ok", encoding="utf-8")
    else:
        print(f"Đã giải nén trước đó: {extract_dir}")

    matches = sorted(extract_dir.rglob(DATA_FILENAME))
    if not matches:
        matches = sorted(
            p
            for p in extract_dir.rglob("*.json")
            if "Logic" in p.name and "Educational" in p.name
        )
    if not matches:
        sample = [str(p.relative_to(extract_dir)) for p in extract_dir.rglob("*.json")][:15]
        raise FileNotFoundError(
            f"Không thấy {DATA_FILENAME} trong zip.\n"
            f"Một số .json trong zip: {sample}"
        )

    src = matches[0]
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, target)
    print(f"Tìm thấy: {src}")
    print(f"Copy → {target}")
    return target


if DATA_SOURCE == "drive":
    _drive_data_path = download_and_extract_from_drive()
    print(f"DATA_SOURCE=drive OK → {_drive_data_path}")
elif DATA_SOURCE == "local":
    print("DATA_SOURCE=local — bỏ qua tải Drive, dùng file trong repo.")
else:
    raise ValueError(f"DATA_SOURCE phải là 'local' hoặc 'drive', nhận: {DATA_SOURCE!r}")

Tải Google Drive (EXACT2026_dataset_2026-05-15.zip)...
  Link: https://drive.google.com/file/d/1-8Wu-2FikPQIX7_rRiY3wfe7c229To2Z/view?usp=drive_link


Downloading...
From: https://drive.google.com/uc?id=1-8Wu-2FikPQIX7_rRiY3wfe7c229To2Z
To: /content/exact_data/_cache/EXACT2026_dataset_2026-05-15.zip
100%|██████████| 356k/356k [00:00<00:00, 117MB/s]

Giải nén → /content/exact_data/_cache/extracted
Tìm thấy: /content/exact_data/_cache/extracted/Logic_Based_Educational_Queries_Text_Only/Logic_Based_Educational_Queries.json
Copy → /content/exact_data/app/data/raw/Logic_Based_Educational_Queries.json
DATA_SOURCE=drive OK → /content/exact_data/app/data/raw/Logic_Based_Educational_Queries.json


In [14]:
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split


def find_data_path_local() -> Path:
    """Tìm JSON trong repo / cwd (không tải Drive)."""
    rels = [
        Path("app/data/raw") / DATA_FILENAME,
        Path("Exact_2026_Laplace-s_Red_Devils/app/data/raw") / DATA_FILENAME,
        Path("data/raw") / DATA_FILENAME,
    ]
    roots = [Path.cwd(), Path(DATA_ROOT).resolve()]
    seen: set[str] = set()
    for start in roots:
        for base in [start, *start.parents]:
            key = str(base.resolve())
            if key in seen:
                continue
            seen.add(key)
            for rel in rels:
                p = (base / rel).resolve()
                if p.is_file():
                    return p
    raise FileNotFoundError(
        f"Không tìm thấy {DATA_FILENAME}\n"
        f"  cwd: {Path.cwd()}\n"
        f"  DATA_ROOT: {DATA_ROOT}\n"
        f"  Gợi ý: đặt DATA_SOURCE='drive' và chạy cell tải Drive phía trên"
    )


if DATA_SOURCE == "drive":
    DATA_PATH = (DATA_ROOT / "app" / "data" / "raw" / DATA_FILENAME).resolve()
    if not DATA_PATH.is_file():
        raise FileNotFoundError(
            f"Chưa có {DATA_PATH}. Chạy cell tải Drive (ngay phía trên) trước."
        )
else:
    DATA_PATH = find_data_path_local()
APP_DIR = DATA_PATH.parent.parent.parent  # app/data/raw → app
ROOT = APP_DIR / "notebooks" / "Logic_Based_Educational_Queries"
if not ROOT.is_dir():
    ROOT = Path.cwd()

OUT_DIR = ROOT / "output" / "pipeline_sft"
PROCESSED_DIR = APP_DIR / "data" / "processed" / "logic_sft"
CHECKPOINT_DIR = OUT_DIR / "final_lora"

OUT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"cwd:      {Path.cwd()}")
print(f"ROOT:     {ROOT}")
print(f"APP_DIR:  {APP_DIR}")
print(f"Data:     {DATA_PATH}")

cwd:      /content
ROOT:     /content
APP_DIR:  /content/exact_data/app
Data:     /content/exact_data/app/data/raw/Logic_Based_Educational_Queries.json


In [15]:
def get_fol(rec: dict) -> list[str]:
    return rec.get("premises-FOL", [])


def normalize_idx(idx: list, n_questions: int) -> list[list]:
    if not idx:
        return [[] for _ in range(n_questions)]
    if all(isinstance(x, int) for x in idx):
        return [list(idx)] if n_questions == 1 else [list(idx)] * n_questions
    return idx


def flatten_records(recs: list[dict]) -> pd.DataFrame:
    """1 dòng = 1 câu hỏi — giống eda_and_preprocessing.ipynb."""
    rows = []
    for rid, rec in enumerate(recs):
        fol_list = get_fol(rec)
        idx_lists = normalize_idx(rec["idx"], len(rec["questions"]))
        for qi, (q, a, exp, idx_used) in enumerate(
            zip(rec["questions"], rec["answers"], rec["explanation"], idx_lists)
        ):
            rows.append(
                {
                    "record_id": rid,
                    "q_idx": qi,
                    "n_premises_used": len(idx_used),
                    "question": q,
                    "answer": a,
                    "explanation": exp,
                    "premises_nl": list(rec["premises-NL"]),
                    "premises_fol": list(fol_list),
                }
            )
    return pd.DataFrame(rows)


def split_record_ids(n_records: int, ratios=(0.8, 0.1, 0.1), seed=42):
    train_r, dev_r, test_r = ratios
    ids = list(range(n_records))
    dev_test = dev_r + test_r
    train_ids, temp = train_test_split(ids, test_size=dev_test, random_state=seed)
    dev_ids, test_ids = train_test_split(
        temp, test_size=test_r / dev_test, random_state=seed
    )
    return sorted(train_ids), sorted(dev_ids), sorted(test_ids)


def _numbered_lines(items: list[str], header: str) -> str:
    if not items:
        return ""
    return header + "\n" + "\n".join(f"{i}. {p}" for i, p in enumerate(items, 1))


def build_user_content(row: dict) -> str:
    blocks = []
    if INCLUDE_FOL_IN_USER and row.get("premises_fol"):
        blocks.append(_numbered_lines(row["premises_fol"], PREMISES_FOL_HEADER))
    if row.get("premises_nl"):
        blocks.append(_numbered_lines(row["premises_nl"], PREMISES_NL_HEADER))
    return USER_TEMPLATE.format(
        premises_block="\n\n".join(blocks).strip(),
        question=row["question"],
    )


def build_assistant_content(row: dict) -> str:
    return ASSISTANT_TEMPLATE.format(
        answer=row["answer"], explanation=row["explanation"]
    )


def build_messages(row: dict) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT.strip()},
        {"role": "user", "content": build_user_content(row)},
        {"role": "assistant", "content": build_assistant_content(row)},
    ]


def _default_splits_export_dir() -> Path:
    if SPLITS_EXPORT_DIR is not None:
        p = Path(SPLITS_EXPORT_DIR)
        return p if p.is_absolute() else Path.cwd() / p
    return (APP_DIR / "data" / "processed" / "logic_sft_splits").resolve()


def export_splits_to_directories(
    splits: dict[str, pd.DataFrame],
    split_meta: dict[str, list[int]],
    export_root: Path,
    *,
    write_sft: bool = True,
) -> dict[str, dict]:
    """Lưu mỗi tập train|dev|test vào export_root/<split>/."""
    export_root = export_root.resolve()
    export_root.mkdir(parents=True, exist_ok=True)

    summary_all = {
        "split_ratios": list(SPLIT_RATIOS),
        "split_seed": SPLIT_SEED,
        "data_source": DATA_SOURCE,
        "data_file": str(DATA_PATH),
        "splits": {},
    }

    for split_name, df_split in splits.items():
        out_dir = export_root / split_name
        out_dir.mkdir(parents=True, exist_ok=True)

        # 1) flat.jsonl — 1 dòng = 1 câu (đủ premise list, question, answer, ...)
        flat_path = out_dir / "flat.jsonl"
        with flat_path.open("w", encoding="utf-8") as f:
            for row in df_split.to_dict(orient="records"):
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

        # 2) sft.jsonl — format chat cho fine-tune
        if write_sft:
            sft_path = out_dir / "sft.jsonl"
            with sft_path.open("w", encoding="utf-8") as f:
                for row in df_split.to_dict(orient="records"):
                    rec = {
                        "record_id": row["record_id"],
                        "q_idx": row["q_idx"],
                        "messages": build_messages(row),
                    }
                    f.write(json.dumps(rec, ensure_ascii=False) + "\n")

        split_summary = {
            "n_records": len(split_meta[split_name]),
            "n_questions": len(df_split),
            "record_ids": split_meta[split_name],
            "files": {
                "flat.jsonl": str(flat_path),
                **({"sft.jsonl": str(out_dir / "sft.jsonl")} if write_sft else {}),
            },
        }
        with (out_dir / "summary.json").open("w", encoding="utf-8") as f:
            json.dump(split_summary, f, indent=2, ensure_ascii=False)

        summary_all["splits"][split_name] = split_summary
        print(f"  {split_name}/ → {len(df_split)} câu @ {out_dir}")

    with (export_root / "split_record_ids.json").open("w", encoding="utf-8") as f:
        json.dump(split_meta, f, indent=2)

    with (export_root / "split_summary.json").open("w", encoding="utf-8") as f:
        json.dump(summary_all, f, indent=2, ensure_ascii=False)

    return summary_all




In [16]:
with open(DATA_PATH, encoding="utf-8") as f:
    records = json.load(f)

df_q = flatten_records(records)
expected = sum(len(r["questions"]) for r in records)
assert len(df_q) == expected == 808, f"Flatten lệch: {len(df_q)} vs {expected}"

train_ids, dev_ids, test_ids = split_record_ids(
    len(records), SPLIT_RATIOS, SPLIT_SEED
)
meta = {"train": train_ids, "dev": dev_ids, "test": test_ids}


def filter_split(name: str) -> pd.DataFrame:
    return df_q[df_q["record_id"].isin(meta[name])].reset_index(drop=True)


splits_df = {k: filter_split(k) for k in meta}
for name, ids in meta.items():
    sub = splits_df[name]
    sub.to_csv(PROCESSED_DIR / f"{name}.csv", index=False)
    print(f"{name}: {len(ids)} records, {len(sub)} câu")

with open(PROCESSED_DIR / "split_record_ids.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print(f"\nTổng: {len(records)} records, {len(df_q)} câu flatten")


if EXPORT_SPLITS:

    _export_root = _default_splits_export_dir()
    print(f"Export splits → {_export_root}\n")
    export_splits_to_directories(splits_df, meta, _export_root)
    print(f"\nDone. Cấu trúc:\n  {_export_root}/train/\n  {_export_root}/dev/\n  {_export_root}/test/")
else:
    print("EXPORT_SPLITS=False — bỏ qua lưu thư mục.")

train: 328 records, 645 câu
dev: 41 records, 81 câu
test: 42 records, 82 câu

Tổng: 411 records, 808 câu flatten


In [17]:
def to_hf_dataset(split_df: pd.DataFrame) -> Dataset:
    rows = []
    for row in split_df.to_dict(orient="records"):
        rows.append(
            {
                "record_id": row["record_id"],
                "q_idx": row["q_idx"],
                "messages": build_messages(row),
            }
        )
    return Dataset.from_list(rows)


dataset_dict = DatasetDict(
    {name: to_hf_dataset(splits_df[name]) for name in ("train", "dev", "test")}
)

sample = dataset_dict["train"][0]
print("Roles:", [m["role"] for m in sample["messages"]])
print("\n--- USER (preview) ---\n", sample["messages"][1]["content"][:800], "...")
print("\n--- ASSISTANT (preview) ---\n", sample["messages"][2]["content"][:400], "...")

Roles: ['system', 'user', 'assistant']

--- USER (preview) ---
 Premises (FOL):
1. ForAll(x, (completed_core_curriculum(x) ∧ passed_science_assessment(x)) → qualified_for_advanced_courses(x))
2. ForAll(x, (qualified_for_advanced_courses(x) ∧ completed_research_methodology(x)) → eligible_for_international_program(x))
3. ForAll(x, passed_language_proficiency(x) → eligible_for_international_program(x))
4. ForAll(x, (eligible_for_international_program(x) ∧ completed_capstone_project(x)) → awarded_honors_diploma(x))
5. ForAll(x, (awarded_honors_diploma(x) ∧ completed_community_service(x)) → qualifies_for_scholarship(x))
6. ForAll(x, (awarded_honors_diploma(x) ∧ received_faculty_recommendation(x)) → qualifies_for_scholarship(x))
7. completed_core_curriculum(Sophia)
8. passed_science_assessment(Sophia)
9. completed_research_methodology(Sophia)
10. completed_c ...

--- ASSISTANT (preview) ---
 Answer: C

Explanation: Premises 7 and 8 confirm Sophia completed the core curriculum and passed the 

## 3. Tokenizer & format chat template

In [18]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=TRUST_REMOTE_CODE
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


dataset_dict = dataset_dict.map(
    format_chat, remove_columns=["messages"]
)
print(dataset_dict)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/645 [00:00<?, ? examples/s]

Map:   0%|          | 0/81 [00:00<?, ? examples/s]

Map:   0%|          | 0/82 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['record_id', 'q_idx', 'text'],
        num_rows: 645
    })
    dev: Dataset({
        features: ['record_id', 'q_idx', 'text'],
        num_rows: 81
    })
    test: Dataset({
        features: ['record_id', 'q_idx', 'text'],
        num_rows: 82
    })
})


## 4. Fine-tune (LoRA r=8, bf16)

In [28]:
if not RUN_TRAIN:
    print("RUN_TRAIN=False — bỏ qua training.")
else:
    import torch
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    from trl import SFTConfig, SFTTrainer

    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        cc = torch.cuda.get_device_capability(0)
        print(f"GPU: {torch.cuda.get_device_name(0)} (compute {cc[0]}.{cc[1]})")
        _probe = torch.zeros(1, device="cuda")
        del _probe

    profile = GPU_PROFILE
    if profile == "auto" and torch.cuda.is_available():
        profile = "kaggle_p100" if torch.cuda.get_device_capability(0)[0] < 7 else "default"
    print(f"GPU_PROFILE dùng cho train: {profile}")

    # P100 16GB: bf16 full 7B + device_map=auto → offload/meta → lỗi CUDA/PEFT
    if profile == "kaggle_p100":
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb_config,
            device_map={"": 0},
            trust_remote_code=TRUST_REMOTE_CODE,
        )
        model = prepare_model_for_kbit_training(model)
        train_bf16, train_fp16 = False, True
    else:
        dtype = torch.bfloat16 if BF16 else torch.float16
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=dtype,
            trust_remote_code=TRUST_REMOTE_CODE,
            device_map={"": 0},
        )
        train_bf16, train_fp16 = BF16, not BF16

    model.config.use_cache = False

    peft_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, peft_config)
    if GRADIENT_CHECKPOINTING:
        # LoRA + checkpoint: bắt buộc use_reentrant=False (tránh CheckpointError)
        model.enable_input_require_grads()
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
    model.print_trainable_parameters()

    sft_config = SFTConfig(
        output_dir=str(OUT_DIR),
        num_train_epochs=NUM_TRAIN_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="cosine",
        weight_decay=WEIGHT_DECAY,
        logging_steps=LOGGING_STEPS,
        logging_strategy="steps",
        logging_first_step=True,
        disable_tqdm=False,  # bật thanh % epoch/step (mặc định True nếu không set)
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        bf16=train_bf16,
        fp16=train_fp16,
        gradient_checkpointing=False,  # đã bật trên PEFT model ở trên
        seed=TRAIN_SEED,
        report_to="none",
        max_length=MAX_SEQ_LENGTH,  # trl>=0.16: max_length (cũ: max_seq_length)
        dataset_text_field="text",
        packing=False,
    )

    trainer_kwargs = dict(
        model=model,
        args=sft_config,
        train_dataset=dataset_dict["train"],
        eval_dataset=dataset_dict["dev"],
    )
    # trl mới: processing_class | trl cũ: tokenizer
    import inspect
    if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters:
        trainer_kwargs["processing_class"] = tokenizer
    else:
        trainer_kwargs["tokenizer"] = tokenizer
    trainer = SFTTrainer(**trainer_kwargs)

    print("Bắt đầu train... (lần đầu có thể im vài phút: load model + tokenize)")
    train_result = trainer.train()
    print(train_result)  # tóm tắt loss / steps sau khi xong
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(CHECKPOINT_DIR))
    tokenizer.save_pretrained(str(CHECKPOINT_DIR))

    metrics = trainer.evaluate()
    with open(OUT_DIR / "train_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    print("Dev metrics:", metrics)

CUDA available: True
GPU: Tesla T4


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 20,185,088 || all params: 7,635,801,600 || trainable%: 0.2643


Adding EOS to train dataset:   0%|          | 0/645 [00:00<?, ? examples/s]

...

Adding EOS to eval dataset:   0%|          | 0/81 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/81 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Bắt đầu train... (lần đầu có thể im vài phút: load model + tokenize)


OutOfMemoryError: CUDA out of memory. Tried to allocate 354.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 111.81 MiB is free. Including non-PyTorch memory, this process has 14.45 GiB memory in use. Of the allocated memory 14.28 GiB is allocated by PyTorch, and 48.90 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 5. Push lên Hugging Face Hub

Repo: **[Laplaces-Red-Devils/logic-v01-fewshot-qwen2.5-7B](https://huggingface.co/Laplaces-Red-Devils/logic-v01-fewshot-qwen2.5-7B)**

### Thông tin cần có trước khi push

| # | Yêu cầu |
|---|--------|
| 1 | Tài khoản HF là **member** org [Laplaces-Red-Devils](https://huggingface.co/Laplaces-Red-Devils) với quyền **write** |
| 2 | **Access Token** (Write) → đặt env `HF_TOKEN` hoặc `HUGGINGFACE_HUB_TOKEN` |
| 3 | Tạo repo model trống trên HF (nếu chưa có): `logic-v01-fewshot-qwen2.5-7B` |
| 4 | Cell train đã chạy xong → có `output/pipeline_sft/final_lora` |

Sau push, pull về:
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
model = AutoModelForCausalLM.from_pretrained("Laplaces-Red-Devils/logic-v01-fewshot-qwen2.5-7B")
tokenizer = AutoTokenizer.from_pretrained("Laplaces-Red-Devils/logic-v01-fewshot-qwen2.5-7B")
```

In [ ]:
if not PUSH_TO_HUB:
    print("PUSH_TO_HUB=False — bỏ qua. Đặt PUSH_TO_HUB=True và HF_TOKEN để push.")
else:
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from huggingface_hub import HfApi

    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    if not token:
        raise EnvironmentError(
            "Thiếu HF_TOKEN. Tạo tại https://huggingface.co/settings/tokens (Write)."
        )

    if not CHECKPOINT_DIR.exists():
        raise FileNotFoundError(f"Chưa có checkpoint: {CHECKPOINT_DIR}")

    print("Merge LoRA...")
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype="auto",
        trust_remote_code=TRUST_REMOTE_CODE,
        device_map="cpu",
    )
    model = PeftModel.from_pretrained(base, str(CHECKPOINT_DIR))
    merged = model.merge_and_unload()

    merged_dir = OUT_DIR / "merged_for_hub"
    merged_dir.mkdir(parents=True, exist_ok=True)
    merged.save_pretrained(merged_dir)

    tok = AutoTokenizer.from_pretrained(
        str(CHECKPOINT_DIR), trust_remote_code=TRUST_REMOTE_CODE
    )
    tok.save_pretrained(merged_dir)

    readme = merged_dir / "README.md"
    readme.write_text(
        f"""---
license: apache-2.0
base_model: {MODEL_NAME}
tags:
  - exact-2026
  - logic
  - qwen2.5
---

# logic-v01-fewshot-qwen2.5-7B

LoRA SFT (r={LORA_R}, bf16) trên Logic_Based_Educational_Queries.
Flatten 808 câu, split 8:1:1 theo record_id.
""",
        encoding="utf-8",
    )

    print(f"Push → {HF_REPO_ID}")
    merged.push_to_hub(
        HF_REPO_ID,
        token=token,
        private=HF_PRIVATE,
        commit_message=HF_COMMIT_MESSAGE,
    )
    tok.push_to_hub(HF_REPO_ID, token=token, private=HF_PRIVATE)
    HfApi(token=token).upload_file(
        path_or_fileobj=str(readme),
        path_in_repo="README.md",
        repo_id=HF_REPO_ID,
        repo_type="model",
    )
    print(f"Done: https://huggingface.co/{HF_REPO_ID}")